# 05 · Random forests and extra trees (bagged ensembles)

Many deep trees on bootstrap samples, averaged: much more stable than one tree, little tuning, no scaling.
Also compares two kinds of feature importance — impurity (built in, biased towards features with many distinct
values) and permutation (how much AUC drops when a feature is shuffled on unseen rows).

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "05"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
RF = lambda **k: RandomForestClassifier(n_estimators=300, random_state=M.SEED, n_jobs=1, **k)
ET = lambda **k: ExtraTreesClassifier(n_estimators=300, random_state=M.SEED, n_jobs=1, **k)
G = lambda name: {f"{name}__max_depth": [3, 5, 8, 12, None], f"{name}__min_samples_leaf": randint(1, 60),
                  f"{name}__max_features": ["sqrt", 0.3, 0.5, 0.8]}
run("RF-01", M.pipeline(M.preprocess("none"), RF()), "FS_ALL", "Random forest, defaults (300 trees)")
run("RF-02", M.tuned(M.pipeline(M.preprocess("none"), RF()), G("randomforestclassifier"), n_iter=12), "FS_ALL", "Random forest, tuned (depth, min leaf, max features)")
run("RF-03", M.tuned(M.pipeline(M.preprocess("none"), ET()), G("extratreesclassifier"), n_iter=12), "FS_ALL", "Extra trees, tuned")
run("RF-04", M.pipeline(M.preprocess("none"), M.TopKPermutation(k=6), RF(min_samples_leaf=20, max_features=0.5)), "FS_ALL", "Top-6 features by permutation importance (inside each fold), then RF")
run("RF-05", M.pipeline(M.preprocess("none"), RF(min_samples_leaf=20, max_features=0.5, class_weight="balanced")), "FS_ALL", "RF (min leaf 20) with balanced class weights")
run("RF-06", M.pipeline(M.preprocess("none"), CalibratedClassifierCV(RF(min_samples_leaf=20, max_features=0.5), method="isotonic", cv=3)), "FS_TRANSFER", "RF (min leaf 20) + isotonic calibration, FS_TRANSFER");

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

In [ ]:
best = board.run_id.iloc[0]
pd.DataFrame([M.corrected_ttest(best, rid) for rid in board.run_id[1:]]).round(4)

## Impurity vs permutation importance (RF, min leaf 20, held-out 25% of training rows)

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
X, y = M.xy(dev, M.FS_ALL); Xf = M.RecencyFill().fit(X).transform(X)
Xa, Xb, ya, yb = train_test_split(Xf, y, test_size=.25, stratify=y, random_state=M.SEED)
rf = RF(min_samples_leaf=20, max_features=0.5).fit(Xa, ya)
pi = permutation_importance(rf, Xb, yb, scoring="roc_auc", n_repeats=20, random_state=M.SEED)
imp = pd.DataFrame({"impurity": rf.feature_importances_, "permutation (AUC drop)": pi.importances_mean}, index=Xf.columns)
imp = imp.sort_values("permutation (AUC drop)", ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 0.3 * len(imp) + 1), sharey=True)
imp["impurity"].plot.barh(ax=ax[0], color="#8DA0CB", title="Impurity importance"); imp["permutation (AUC drop)"].plot.barh(ax=ax[1], color="#66C2A5", title="Permutation importance")
ax[0].invert_yaxis(); plt.tight_layout(); plt.show()

## Findings (Salud-tuned world)
- **Best family on cross-validation:** RF-05 (min leaf 20, balanced weights) reaches **0.887**. RF-02 (tuned) and RF-04 (top-6 features only) reach 0.884, which is not significantly different (p ≥ 0.19).
- **Six features are enough:** RF-04 matches the full forest. This is the "train only on the important ones" result.
- The gain over logistic regression is about 0.01 AUC: real, but small.